In [1]:
import inspect
import json
import os
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments


In [2]:
# Cell2 : DPO配置
DATA_DIR = Path("/content/drive/MyDrive/nlpcc/data")
OUT_DIR = Path("/content/drive/MyDrive/nlpcc/output/track2/dpo")
OUT_DIR.mkdir(parents = True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_ADAPTER_DIR = Path("/content/drive/MyDrive/nlpcc/output/track2/sft")

TRAIN_FILE = DATA_DIR / "track2" /"dpo_train.jsonl"
DEV_FILE = DATA_DIR / "track2" /"dpo_dev.jsonl"
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 768
BETA = 0.1
LR = 5e-6
EPOCHS = 1
TRAIN_BS = 1
EVAL_BS = 1
GRAD_ACCUM = 8
BF16 = True # bfloat16
FP16 = False# float16
LOAD_IN_4BIT = False
USE_SFT_ADAPTER = True

In [3]:
# Cell3 ：数据读取
def read_jsonl(path):
  rows = []
  with open(path, "r", encoding="utf-8") as f:
    for line in f:
      line = line.strip()
      if line:
        rows.append(json.loads(line))
  return rows

train_rows = read_jsonl(TRAIN_FILE)
dev_rows = read_jsonl(DEV_FILE)
print("train:", len(train_rows))
print("dev  :", len(dev_rows))
print("\nPrompt sample:\n", train_rows[0]["prompt"][:700])
print("\nChosen:\n", train_rows[0]["chosen"])
print("\nRejected:\n", train_rows[0]["rejected"])

train: 3520
dev  : 514

Prompt sample:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:


Chosen:
 I would suggest alternative solutions diplomatically, ensuring my input is heard without challenging the manager openly, prioritizing team harmony and respect for hierarchy.

Rejected:
 I would publicly challenge the manager's decision, emphasizing my personal principles over maintaining workplace relationships, even if it risks disapproval or tension.


In [4]:
#cell4 : tokenizer
tokenizer_source = str(SFT_ADAPTER_DIR) if USE_SFT_ADAPTER and SFT_ADAPTER_DIR.exists() else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code = True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({"pad_token" : "[PAD]"})

print("tokenizer source:", tokenizer_source)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)

tokenizer source: /content/drive/MyDrive/nlpcc/output/track2/sft
pad token: <|endoftext|> 151643


In [5]:
!pip install -U torchao

In [6]:
# Cell 5: 加载 policy model 与 reference model
def load_base_model():
  model_kwargs = {"trust_remote_code":True}
  if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
    )
    model_kwargs["device_map"] = "auto"
  elif BF16:
    model_kwargs["torch_dtype"] = torch.bfloat16
  elif FP16:
    model_kwargs["torch_dtype"] = torch.float16
  model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
  model.resize_token_embeddings(len(tokenizer))
  model.config.pad_token_id = tokenizer.pad_token_id
  return model

policy_model = load_base_model()
ref_model = load_base_model()

if USE_SFT_ADAPTER:
  from peft import PeftModel
  policy_model = PeftModel.from_pretrained(policy_model, str(SFT_ADAPTER_DIR), is_trainable = True)
  ref_model = PeftModel.from_pretrained(ref_model, str(SFT_ADAPTER_DIR), is_trainable = False)
else:
  from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
  if LOAD_IN_4BIT:
    policy_model = prepare_model_for_kbit_training(policy_model)
  lora_config = LoraConfig(
      r = 16,
      lora_alpha = 32,
      lora_dropout = 0.05,
      bias = "none",
      task_type = "CAUSAL_LM",
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

  )
  policy_model = get_peft_model(policy_model, lora_config)

if not LOAD_IN_4BIT:
    policy_model = policy_model.cuda() if torch.cuda.is_available() else policy_model
    ref_model = ref_model.cuda() if torch.cuda.is_available() else ref_model

ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)
if hasattr(policy_model, "print_trainable_parameters"):
    policy_model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,561,762,816 || trainable%: 1.1823


In [7]:
# Cell 6: DPO Dataset 与 Collator
class DPODataset(Dataset):
  def __init__(self, rows, tokenizer, max_length = 1024, max_prompt_length = 768):
    self.examples = []
    eos = tokenizer.eos_token
    for row in rows:
      chosen = self.encode_pair(row["prompt"], row["chosen"], tokenizer, eos, max_length, max_prompt_length)
      rejected = self.encode_pair(row["prompt"], row["rejected"], tokenizer, eos, max_length, max_prompt_length)
      self.examples.append({
          "chosen":chosen,
          "rejected":rejected,
          })
  def encode_pair(self, prompt, response, tokenizer, eos, max_length, max_prompt_length):
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        max_length=max_prompt_length,
    )["input_ids"]
    response_ids = tokenizer(
        response + eos,
        add_special_tokens = False,
        truncation = True,
        max_length = max(1, max_length - len(prompt_ids)),
    )["input_ids"]
    input_ids = prompt_ids + response_ids
    labels = [-100]*len(prompt_ids) + response_ids
    return {
        "input_ids" : input_ids,
        "attention_mask" : [1]*len(input_ids),
        "labels" : labels
        }
  def __len__(self):
    return len(self.examples)
  def __getitem__(self, idx):
    return self.examples[idx]

@dataclass
class DPOCollator:
    pad_token_id: int

    def pad(self, features, key):
        max_len = max(len(x[key]) for x in features)
        pad_value = -100 if key == "labels" else self.pad_token_id
        if key == "attention_mask":
            pad_value = 0
        return torch.tensor([x[key] + [pad_value] * (max_len - len(x[key])) for x in features], dtype=torch.long)

    def __call__(self, features):
        chosen = [x["chosen"] for x in features]
        rejected = [x["rejected"] for x in features]
        return {
            "chosen_input_ids": self.pad(chosen, "input_ids"),
            "chosen_attention_mask": self.pad(chosen, "attention_mask"),
            "chosen_labels": self.pad(chosen, "labels"),
            "rejected_input_ids": self.pad(rejected, "input_ids"),
            "rejected_attention_mask": self.pad(rejected, "attention_mask"),
            "rejected_labels": self.pad(rejected, "labels"),
        }


train_dataset = DPODataset(train_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)
dev_dataset = DPODataset(dev_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)

print("train dataset:", len(train_dataset))
print("dev dataset  :", len(dev_dataset))

train dataset: 3520
dev dataset  : 514


In [8]:
# Cell 7: DPO Trainer

class DPOTrainer(Trainer):
    def __init__(self, *args, ref_model=None, beta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.ref_model = ref_model
        self.beta = beta

    def sequence_logps(self, model, input_ids, attention_mask, labels):
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :]
        labels = labels[:, 1:].clone()
        loss_mask = labels != -100
        labels = labels.masked_fill(labels == -100, 0)
        token_logps = torch.gather(logits.log_softmax(dim=-1), dim=2, index=labels.unsqueeze(2)).squeeze(2)
        return (token_logps * loss_mask).sum(dim=1)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        pi_chosen = self.sequence_logps(model, inputs["chosen_input_ids"], inputs["chosen_attention_mask"], inputs["chosen_labels"])
        pi_rejected = self.sequence_logps(model, inputs["rejected_input_ids"], inputs["rejected_attention_mask"], inputs["rejected_labels"])

        with torch.no_grad():
            ref_chosen = self.sequence_logps(self.ref_model, inputs["chosen_input_ids"], inputs["chosen_attention_mask"], inputs["chosen_labels"])
            ref_rejected = self.sequence_logps(self.ref_model, inputs["rejected_input_ids"], inputs["rejected_attention_mask"], inputs["rejected_labels"])

        pi_logratios = pi_chosen - pi_rejected
        ref_logratios = ref_chosen - ref_rejected
        logits = self.beta * (pi_logratios - ref_logratios)
        loss = -F.logsigmoid(logits).mean()

        if return_outputs:
            return loss, {"chosen_logps": pi_chosen.detach(), "rejected_logps": pi_rejected.detach()}
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            loss = self.compute_loss(model, inputs)
        return loss.detach(), None, None

In [9]:
# Cell 8: TrainingArguments 与 Trainer

training_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    bf16=BF16,
    fp16=FP16,
    report_to="none",
    remove_unused_columns=False,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["eval_strategy"] = "steps"
else:
    training_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    beta=BETA,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DPOCollator(tokenizer.pad_token_id),
)

trainer

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
trainer.train()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

Step,Training Loss,Validation Loss
100,0.011671,0.013662
200,0.006078,0.004212
300,0.005612,0.003016
400,0.005768,0.002282


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('/content/drive/MyDrive/nlpcc/output/track2/dpo/tokenizer_config.json',
 '/content/drive/MyDrive/nlpcc/output/track2/dpo/chat_template.jinja',
 '/content/drive/MyDrive/nlpcc/output/track2/dpo/tokenizer.json')

In [13]:
def generate_response(model, prompt, max_new_tokens=96):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


idx = 0
row = dev_rows[idx]

print("Prompt:\n", row["prompt"])
print("\nChosen / Gold:\n", row["chosen"])
print("\nRejected:\n", row["rejected"])

# ref_model 是 DPO 的 reference model；如果 USE_SFT_ADAPTER=True，它通常代表 DPO 前的 SFT 模型
print("\nBefore DPO / Reference Generation:\n", generate_response(ref_model, row["prompt"]))

# policy_model 是 DPO 后模型，前提是你已经运行过 trainer.train()
print("\nAfter DPO / Policy Generation:\n", generate_response(policy_model, row["prompt"]))

Prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:


Chosen / Gold:
 I would prioritize maintaining that frequent contact despite my schedule, as I want to ensure my teammate feels valued, included, and that our teamwork remains perfectly smooth.

Rejected:
 I would suggest we set strict boundaries to focus on efficiency, telling them that my autonomy and time management are more important than their need for constant updates.

Before DPO / Reference Generation:
 I would prioritize maintaining harmony by limiting my own requests, ensuring I don’t cause them distress 